<a href="https://colab.research.google.com/github/Sargam-max/Machine_Learning/blob/main/Logistic%20regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
##Logistic regression using pandas and numpy

In [1]:
import pandas as pd
import numpy as np

In [16]:
data=pd.DataFrame({
    'hours': [1, 2, 3, 4, 5, 6, 7],
    'pass': [0, 0, 0, 1, 1, 1, 1]

})
X=data[['hours']]
y=data['pass']

##Add bias column
X = np.column_stack((np.ones(len(X)), X))

print(X)


[[1. 1.]
 [1. 2.]
 [1. 3.]
 [1. 4.]
 [1. 5.]
 [1. 6.]
 [1. 7.]]


In [21]:
weights = np.zeros(2)
print(weights)

[0. 0.]


In [18]:
learning_rate = 0.01
epochs = 1000

In [27]:
from numpy._core.defchararray import count

for i in range(epochs):
  # Linear equation
    z = np.dot(X, weights)
   # print("z",z)

# Sigmoid
    predictions = 1 / (1 + np.exp(-z))
# Error
    error = predictions - y
# Gradient
    gradient = np.dot(X.T, error) / len(y)
# Update weights
    weights = weights - learning_rate * gradient

In [28]:
z = np.dot(X, weights)
predictions = 1 / (1 + np.exp(-z))

In [29]:
result = (predictions >= 0.5).astype(int)

In [30]:
print("Predicted:")
print(result)

Predicted:
[0 0 0 1 1 1 1]


Dataset
↓
Select X and y
↓
Add bias column
↓
Initialize weights = [0,0]
↓
Repeat:
    z = X·w
    sigmoid(z)
    calculate error
    update weights
↓
Get final prediction


The Core Mechanism1.Start with the Linear Equation:

Step

1.Just like linear regression, the model starts by calculating a weighted sum of the input features. If you have inputs $x_1, x_2$, it calculates a raw score ($z$):$$z = \beta_0 + \beta_1x_1 + \beta_2x_2$$Where $\beta_0$ is the bias (intercept) and $\beta_1, \beta_2$ are the weights (coefficients). The problem is that $z$ can be any number from negative infinity to positive infinity.

2.Squish it with the Sigmoid Function:Step

2.To turn that unrestricted score $z$ into a neat probability between 0 and 1, we pass it through the Sigmoid (or Logistic) function.The Sigmoid function formula is:$$\sigma(z) = \frac{1}{1 + e^{-z}}$$No matter how massive or microscopic $z$ is, this function squashes it into a curve that strictly outputs a value between 0 and 1. This output ($p$) is your probability.

3.Apply a Decision Threshold:Step

3.Now that you have a probability $p$, you need to make a final binary choice. By default, the model uses a threshold of 0.5:If $p \ge 0.5$, predict 1 (Yes/True).If $p < 0.5$, predict 0 (No/False).


4.Measure the Error (Log Loss):
Step 4.During training, the model needs to know how wrong its guesses are. We can't use standard Mean Squared Error here because the curve makes it inefficient for optimization. Instead, we use Log Loss (Binary Cross-Entropy):$$\text{Cost} = -[y \log(p) + (1 - y) \log(1 - p)]$$This penalizes the model heavily if it is confident and wrong (e.g., predicting a 99% chance of 1, but the real answer is 0).


5.Optimize with Gradient Descent:Step

5.To minimize that Log Loss, the model uses an optimization algorithm called Gradient Descent. It calculates the derivative of the loss function, figures out which direction reduces the error, and subtly tweaks the weights ($\beta$) and bias ($\beta_0$). This loop repeats across your dataset until the weights stop changing significantly.


Why Not Just Use Linear Regression?If you try to fit a straight line to binary data, two things go wrong:Out of bounds: A straight line will eventually predict values below 0 or above 1, which makes no sense for probabilities.Extreme sensitivity: A single extreme data point (an outlier) far to one side can drastically tilt your linear line, completely ruining your predictions closer to the center.

In [1]:
import numpy as np
import pandas as pd

# STEP 1: DOWNLOAD AND CLEAN THE DATA

print("Step 1: Loading and Cleaning Data")

# Download the dataset directly from the web
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']
df = pd.read_csv(url, names=columns)

--- Step 1: Loading and Cleaning Data ---


In [2]:
df['species'] = df['species'].map({'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2})

In [3]:
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()
y = df['species'].to_numpy()

In [5]:
print(f"Dataset ready! Total flowers: {len(X)} (50 of each species)")

Dataset ready! Total flowers: 150 (50 of each species)


In [6]:
# STEP 2: SPLIT THE DATA INTO TRAIN AND TEST SETS

In [7]:
print("\n--- Step 2: Splitting Data into Train (80%) and Test (20%) ---")

np.random.seed(7)
indices = np.arange(len(X))
np.random.shuffle(indices)
X = X[indices]
y = y[indices]


--- Step 2: Splitting Data into Train (80%) and Test (20%) ---


In [8]:
# 80% of 150 flowers = 120 training items, 30 testing items
train_size = 120

X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"Training set size: {len(X_train)} flowers")
print(f"Testing set size: {len(X_test)} flowers")

Training set size: 120 flowers
Testing set size: 30 flowers


In [10]:
# STEP 3: INITIALIZE OUR PARAMETERS FOR 3 SEPARATE MODELS
print("\n--- Step 3: Initializing Weights and Biases for 3 Classes ---")

num_features = 4
num_classes = 3

# We need a set of weights and a bias for EACH of our 3 classes.
# We will store them in matrices/lists so we can loop through them easily.
weights = np.zeros((num_classes, num_features)) # Matrix of size 3x4
biases = np.zeros(num_classes)                  # Array of size 3

learning_rate = 0.05
epochs = 500

print("Starting weights initialized to 0 for all 3 classes.")


--- Step 3: Initializing Weights and Biases for 3 Classes ---
Starting weights initialized to 0 for all 3 classes.


In [11]:
# STEP 4: THE TRAINING LOOP (One-vs-Rest Gradient Descent)
# =====================================================================
print("\n--- Step 4: Training the 3 Models Simultaneously ---")

num_samples = len(X_train)

for epoch in range(epochs):

    # We track separate gradients for each class model
    total_grad_w = np.zeros((num_classes, num_features))
    total_grad_b = np.zeros(num_classes)
    total_cost = 0.0

    # Loop through every single flower in our training set
    for i in range(num_samples):
        features = X_train[i]   # Array of 4 numbers [sepal_l, sepal_w, petal_l, petal_w]
        actual_class = y_train[i] # The true answer (0, 1, or 2)

        # Train each of the 3 models on this specific flower
        for c in range(num_classes):
            # 1. Calculate raw score z for model 'c'
            z = np.dot(weights[c], features) + biases[c]

            # 2. Get binary probability for model 'c'
            predicted_prob = 1 / (1 + np.exp(-z))

            # 3. Determine the temporary "binary" true answer for this model.
            # If the flower is actually species 'c', the target for this specific model is 1.
            # If the flower is ANY other species, the target for this model is 0.
            binary_true_answer = 1.0 if (actual_class == c) else 0.0

            # 4. Calculate error for this specific model
            error = predicted_prob - binary_true_answer

            # 5. Accumulate gradients for this model's weights and bias
            total_grad_w[c] += error * features
            total_grad_b[c] += error

            # 6. Accumulate cost (only for tracking progress)
            if actual_class == c:
                p_clipped = np.clip(predicted_prob, 1e-15, 1 - 1e-15)
                total_cost += -np.log(p_clipped)

    # After looking at all samples, calculate averages and update parameters
    for c in range(num_classes):
        avg_grad_w = total_grad_w[c] / num_samples
        avg_grad_b = total_grad_b[c] / num_samples

        # Gradient descent step
        weights[c] = weights[c] - (learning_rate * avg_grad_w)
        biases[c] = biases[c] - (learning_rate * avg_grad_b)

    # Print progress status
    if epoch % 100 == 0:
        avg_cost = total_cost / num_samples
        print(f"Epoch {epoch:3d} | Average Multi-Class Loss: {avg_cost:.4f}")


--- Step 4: Training the 3 Models Simultaneously ---
Epoch   0 | Average Multi-Class Loss: 0.6931
Epoch 100 | Average Multi-Class Loss: 0.5391
Epoch 200 | Average Multi-Class Loss: 0.4806
Epoch 300 | Average Multi-Class Loss: 0.4520
Epoch 400 | Average Multi-Class Loss: 0.4330


In [12]:
# STEP 5: EVALUATE ON UNSEEN MULTI-CLASS DATA
# =====================================================================
print("\n--- Step 5: Evaluating on Unseen Test Data ---")

correct_predictions = 0

# Loop through our 30 test flowers
for i in range(len(X_test)):
    features = X_test[i]
    true_answer = y_test[i]

    # Store the output probability of all three models
    class_probabilities = []

    for c in range(num_classes):
        z = np.dot(weights[c], features) + biases[c]
        prob = 1 / (1 + np.exp(-z))
        class_probabilities.append(prob)

    # CRITICAL MULTICLASS STEP:
    # Pick the index of the highest probability (0, 1, or 2) using np.argmax
    final_decision = np.argmax(class_probabilities)

    if final_decision == true_answer:
        correct_predictions += 1
        result_text = "CORRECT"
    else:
        result_text = "WRONG"

    # Format the probabilities nicely for printing
    prob_strings = [f"Class {c}: {p*100:4.1f}%" for c, p in enumerate(class_probabilities)]
    print(f"Flower #{i+1:2d}: True={true_answer} | Probs=[{', '.join(prob_strings)}] -> Decision={final_decision} [{result_text}]")

# Print final score
final_accuracy = (correct_predictions / len(X_test)) * 100
print(f"\nFinal Multiclass Test Accuracy Score: {final_accuracy:.1f}%")


--- Step 5: Evaluating on Unseen Test Data ---
Flower # 1: True=0 | Probs=[Class 0: 94.7%, Class 1: 20.3%, Class 2:  0.1%] -> Decision=0 [CORRECT]
Flower # 2: True=1 | Probs=[Class 0:  8.7%, Class 1: 29.1%, Class 2: 12.6%] -> Decision=1 [CORRECT]
Flower # 3: True=1 | Probs=[Class 0:  3.0%, Class 1: 30.8%, Class 2: 25.5%] -> Decision=1 [CORRECT]
Flower # 4: True=2 | Probs=[Class 0:  0.5%, Class 1: 40.5%, Class 2: 61.1%] -> Decision=2 [CORRECT]
Flower # 5: True=2 | Probs=[Class 0:  0.2%, Class 1: 37.9%, Class 2: 73.3%] -> Decision=2 [CORRECT]
Flower # 6: True=1 | Probs=[Class 0:  1.9%, Class 1: 44.0%, Class 2: 38.8%] -> Decision=1 [CORRECT]
Flower # 7: True=2 | Probs=[Class 0:  0.5%, Class 1: 43.8%, Class 2: 56.8%] -> Decision=2 [CORRECT]
Flower # 8: True=0 | Probs=[Class 0: 91.2%, Class 1: 17.8%, Class 2:  0.2%] -> Decision=0 [CORRECT]
Flower # 9: True=1 | Probs=[Class 0:  1.7%, Class 1: 29.4%, Class 2: 29.9%] -> Decision=2 [WRONG]
Flower #10: True=1 | Probs=[Class 0:  3.7%, Class 1: 4